# 02 — Frame-Event Extraction & Repo-Month Panel  
### *Coding Alone* · ICSE 2027 SEIS pipeline

**Self-contained.** Set PROJECT_ID (same as before) → **Runtime → Run all**.

This notebook does the **one expensive scan** of the project: it extracts the rich per-event detail (who did what, on which PR/issue, with how many commits) for your ~50k **frame repos** across the whole window, into a compact table `frame_events_<yr>` that you own. After this, every metric query reads *your* small table — GitHub Archive is never scanned again.

**⚠ Cost:** this touches GH Archive's large `payload` column, so it's the biggest scan in the project. **Run the COST PREVIEW cell first** — it dry-runs everything for free and prints the exact TB and $ before you spend anything. Then decide whether to run the extraction.

**Interruption-safe:** one idempotent, overwrite table per year + a Drive manifest; re-running skips finished years and reattaches to jobs still running server-side.

Steps: setup → load frame → **cost preview (free)** → extract per year → repo-month rollup → sanity.


## Config — edit PROJECT_ID (same id as before)

In [ ]:
# ==== EDIT ME ==============================================================
PROJECT_ID = "YOUR-GCP-PROJECT-ID"
# ==========================================================================
WINDOW_START = "201801"; WINDOW_END = "202606"
BQ_DATASET = "seis"; BQ_LOCATION = "US"
MAX_SCAN_GB = 6000            # per-YEAR cap; the payload column is big. Preview shows real cost.
DRIVE_BASE = "/content/drive/MyDrive/ICSE_SEIS"
assert PROJECT_ID != "PASTE-YOUR-PROJECT-ID-HERE", "Set PROJECT_ID first."
print("Config loaded. PROJECT_ID =", PROJECT_ID)


## Setup — auth, BigQuery, Drive, helpers
Sign-in popup appears here — choose **ANON@institution**.

In [ ]:
import os, json, time, functools
import pandas as pd, numpy as np
from google.colab import auth, drive
from google.cloud import bigquery

auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
drive.mount('/content/drive')
for sub in ["", "/data", "/data/derived", "/figures", "/tables", "/checkpoints"]:
    os.makedirs(DRIVE_BASE + sub, exist_ok=True)
CKPT_DIR = f"{DRIVE_BASE}/checkpoints"; os.makedirs(CKPT_DIR, exist_ok=True)

def _atomic_write_text(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as f: f.write(text); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
def load_json(name, default):
    p = f"{CKPT_DIR}/{name}"
    return json.load(open(p)) if os.path.exists(p) else default
def save_json(name, obj): _atomic_write_text(f"{CKPT_DIR}/{name}", json.dumps(obj, indent=2, default=str))
def manifest_done(name): return set(load_json(f"manifest_{name}.json", []))
def manifest_mark(name, key):
    s = manifest_done(name); s.add(str(key)); save_json(f"manifest_{name}.json", sorted(s)); return s
_job_records = load_json("bq_jobs.json", {})
def _save_job_records(): save_json("bq_jobs.json", _job_records)

def estimate_gb(sql):
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    return dry.total_bytes_processed / 1e9

def run_query(sql, destination=None, write="WRITE_TRUNCATE", override=False,
              to_dataframe=False, label=None, job_key=None, cluster=None):
    tag = f"[{label}] " if label else ""
    if job_key and job_key in _job_records:
        rec = _job_records[job_key]
        try:
            prev = client.get_job(rec["job_id"], location=rec.get("location", BQ_LOCATION))
            if prev.state == "DONE" and prev.error_result is None:
                print(f"{tag}Reusing finished job {rec['job_id']}.")
                r = prev.result(); return r.to_dataframe() if to_dataframe else r
        except Exception as e: print(f"{tag}(reattach failed: {e}; resubmitting)")
    gb = estimate_gb(sql)
    print(f"{tag}Will scan ~{gb:,.1f} GB (~${gb/1000*6.25:,.2f})")
    if gb > MAX_SCAN_GB and not override:
        raise RuntimeError(f"ABORT: {gb:,.1f} GB > MAX_SCAN_GB={MAX_SCAN_GB}. Pass override=True to force.")
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=int(MAX_SCAN_GB*1e9*1.05))
    if destination:
        cfg.destination = f"{PROJECT_ID}.{BQ_DATASET}.{destination}"; cfg.write_disposition = write
    if cluster: cfg.clustering_fields = cluster
    job = client.query(sql, job_config=cfg)
    if job_key: _job_records[job_key] = {"job_id": job.job_id, "location": BQ_LOCATION}; _save_job_records()
    r = job.result()
    print(f"{tag}Done. Scanned {job.total_bytes_processed/1e9:,.1f} GB" + (f" -> {destination}" if destination else ""))
    return r.to_dataframe() if to_dataframe else r

YEARS = list(range(int(WINDOW_START[:4]), int(WINDOW_END[:4]) + 1))
print("Ready. Years:", YEARS, "| frame-event years done:", sorted(manifest_done('frame_events')))


## Confirm the sampling frame exists
Quick check that `seis.sampling_frame` from notebook 01 is present and how many repos it has.

In [ ]:
fr = client.query(f"SELECT COUNT(*) n FROM `{PROJECT_ID}.{BQ_DATASET}.sampling_frame`").result().to_dataframe()
print("Frame repos:", int(fr.n.iloc[0]))
assert int(fr.n.iloc[0]) > 0, "sampling_frame is empty -- run notebook 01 first."


## The extraction query
For each frame repo, keep the collaboration-relevant events and parse the payload into tidy columns: the PR/issue `number`, the `action`, the thread's `target_author` (who is being replied to — needed for reciprocity & networks), the `commit_count` (from PushEvent), and a `is_bot` flag (so we can separate human interaction from automation).

In [ ]:
EVENT_TYPES = ('PushEvent','PullRequestEvent','PullRequestReviewEvent',
               'PullRequestReviewCommentEvent','IssuesEvent','IssueCommentEvent',
               'CommitCommentEvent','WatchEvent','ForkEvent')

def frame_events_sql(yr):
    start = f"{yr}01"; end = f"{yr}12" if yr != YEARS[-1] else min(f"{yr}12", WINDOW_END)
    types = ",".join(f"'{t}'" for t in EVENT_TYPES)
    return f"""
    SELECT
      f.repo_id, e.created_at,
      FORMAT_TIMESTAMP('%Y%m', e.created_at) AS ym,
      e.type, e.actor.id AS actor_id, e.actor.login AS actor_login,
      (e.actor.login LIKE '%[bot]') AS is_bot,
      JSON_EXTRACT_SCALAR(e.payload,'$.action') AS action,
      CAST(COALESCE(JSON_EXTRACT_SCALAR(e.payload,'$.pull_request.number'),
                    JSON_EXTRACT_SCALAR(e.payload,'$.issue.number'),
                    JSON_EXTRACT_SCALAR(e.payload,'$.number')) AS INT64) AS number,
      COALESCE(JSON_EXTRACT_SCALAR(e.payload,'$.pull_request.user.login'),
               JSON_EXTRACT_SCALAR(e.payload,'$.issue.user.login')) AS target_author,
      CAST(JSON_EXTRACT_SCALAR(e.payload,'$.distinct_size') AS INT64) AS commit_count
    FROM `githubarchive.month.*` e
    JOIN `{PROJECT_ID}.{BQ_DATASET}.sampling_frame` f ON e.repo.id = f.repo_id
    WHERE e._TABLE_SUFFIX BETWEEN '{start}' AND '{end}' AND e.type IN ({types})
    """
print("extraction query builder ready.")


## 💲 COST PREVIEW — run this FIRST (free)
Dry-runs every year and prints the **total TB and $** the extraction will cost. This spends nothing. Look at the total, then decide whether to run the extraction cell below.

In [ ]:
total_gb = 0.0
for yr in YEARS:
    if str(yr) in manifest_done('frame_events'):
        print(f"{yr}: already extracted — skip"); continue
    gb = estimate_gb(frame_events_sql(yr)); total_gb += gb
    print(f"{yr}: ~{gb:,.0f} GB  (~${gb/1000*6.25:,.2f})")
print("-"*48)
print(f"TOTAL to extract: ~{total_gb/1000:,.2f} TB  (~${total_gb/1000*6.25:,.2f})")
print("^ dry-run estimate, costs $0. Run the next cell only if you're OK with this.")


## Extract — per year, idempotent, clustered by repo_id
Runs only after you've seen the preview. Each year overwrites its own `frame_events_<yr>` table and is checkpointed. Interrupt anytime; re-run resumes. To stop early, just interrupt — finished years stay saved.

In [ ]:
for yr in YEARS:
    if str(yr) in manifest_done('frame_events'):
        print(f"{yr}: already done — skip"); continue
    run_query(frame_events_sql(yr), destination=f"frame_events_{yr}",
              write="WRITE_TRUNCATE", cluster=["repo_id"],
              label=f"frame-events-{yr}", job_key=f"frame_events_{yr}")
    manifest_mark("frame_events", yr)
    print(f"{yr}: extracted & checkpointed")
print("All frame-event years done:", sorted(manifest_done('frame_events')))


## Repo-month panel (cheap — reads only your extracted tables)
Rolls `frame_events_*` up to one row per repo-month with the counts the isolation metrics need, including **human-only** variants (bots excluded) for the interaction signals. This is the backbone panel for notebook 03.

In [ ]:
panel_sql = f"""
SELECT repo_id, ym,
  SUM(IF(type='PushEvent', IFNULL(commit_count,0), 0))              AS commits,
  COUNTIF(type='PushEvent')                                         AS pushes,
  COUNTIF(type='PullRequestEvent' AND action='opened')             AS pr_opened,
  COUNTIF(type='IssuesEvent'      AND action='opened')             AS issues_opened,
  COUNTIF(type='IssueCommentEvent')                                AS issue_comments,
  COUNTIF(type='IssueCommentEvent' AND NOT is_bot)                 AS issue_comments_human,
  COUNTIF(type='PullRequestReviewCommentEvent')                    AS review_comments,
  COUNTIF(type='PullRequestReviewCommentEvent' AND NOT is_bot)     AS review_comments_human,
  COUNTIF(type='PullRequestReviewEvent')                           AS reviews,
  COUNTIF(type='CommitCommentEvent')                               AS commit_comments,
  COUNTIF(type='WatchEvent')                                       AS stars,
  COUNTIF(type='ForkEvent')                                        AS forks,
  COUNT(DISTINCT actor_id)                                         AS distinct_actors,
  COUNT(DISTINCT IF(NOT is_bot, actor_id, NULL))                   AS distinct_actors_human
FROM `{PROJECT_ID}.{BQ_DATASET}.frame_events_*`
GROUP BY repo_id, ym
"""
run_query(panel_sql, destination="repo_month_events", label="repo-month-panel",
          cluster=["repo_id"], job_key="repo_month_panel")
print("repo_month_events built.")


## Sanity check
Row counts and a peek at the panel.

In [ ]:
s = client.query(f"""
SELECT COUNT(*) AS repo_months, COUNT(DISTINCT repo_id) AS repos,
       MIN(ym) AS first_ym, MAX(ym) AS last_ym,
       ROUND(AVG(issue_comments_human+review_comments_human),2) AS avg_human_comments,
       ROUND(AVG(commits),2) AS avg_commits
FROM `{PROJECT_ID}.{BQ_DATASET}.repo_month_events`
""").result().to_dataframe()
display(s)
peek = client.query(f"SELECT * FROM `{PROJECT_ID}.{BQ_DATASET}.repo_month_events` ORDER BY repo_id, ym LIMIT 12").result().to_dataframe()
display(peek)
manifest_mark("stages", "02_population_events")
print("Stage 02 complete.")


## Done ✅
You now have `frame_events_*` (rich per-event detail, extracted once) and `repo_month_events` (the repo-month panel). All future notebooks read these — no more GH Archive scans.

Next: **`03_metrics_population.ipynb`** — compute the isolation metrics (Collaboration Density, Solitary Workflow Ratio, Communication Reciprocity, Human Interaction Entropy) from these tables.